# ResNet-8 HPO on Colab

Thin driver: all logic lives in the repo's `.py` files, so Colab and the Mac run identical code.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Keep this tab open (idle tabs get disconnected).

In [ ]:
# Which seeds this machine runs. Hybrid plan: Mac does 42, 43; Colab does 44, 45, 46.
SEEDS = "44 45 46"
BRANCH = "dev"
OUT = "/content/drive/MyDrive/ResNet-8-results"

## 1. Check the GPU
Should show a Tesla T4. If it says *command not found*, the runtime has no GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Mount Google Drive
`/content` is wiped when the session ends; Drive survives. Results go to Drive, everything else stays local.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p {OUT}

## 3. Get the code
Clones into local `/content` (fast disk). The dataset is downloaded next to it on the first run (~10 s).
Colab's preinstalled torch/torchvision are used; `requirements.txt` is only for the Mac.

In [ ]:
import os
if os.path.exists("/content/ResNet-8"):
    !git -C /content/ResNet-8 pull
else:
    !git clone -b {BRANCH} https://github.com/tomsmf/ResNet-8.git /content/ResNet-8
%cd /content/ResNet-8
!git log --oneline -1

## 4. Quick speed test (optional, ~1 min)
Two 2-epoch evaluations into a throwaway folder. Multiply the per-eval time by 5 for a 10-epoch estimate.

In [ ]:
!python run_search.py --method random --seeds 42 --budget 2 --epochs 2 --out /content/speedtest

## 5. Start both searches in the background
`nohup ... &` detaches the processes from the cell, so they keep running while you run other cells.
Each finished evaluation is appended to Drive immediately.

**After a disconnect:** run cells 1–3 and this cell again. Finished evaluations are skipped automatically.

In [ ]:
import subprocess
for method in ["grid", "random"]:
    # Guard: never start a second process for the same method (it would redo the same evaluations).
    if subprocess.run(["pgrep", "-f", "--", f"--method {method}"], capture_output=True).stdout:
        print(f"{method} is already running, not starting it again")
        continue
    !nohup python run_search.py --method {method} --seeds {SEEDS} --out {OUT} >> {OUT}/{method}_colab.log 2>&1 &
!sleep 5; ps aux | grep [r]un_search

## 6. Watch progress
Re-run any time. Shows the last lines of each log and the number of finished evaluations per file.

In [ ]:
!tail -n 3 {OUT}/grid_colab.log {OUT}/random_colab.log
!wc -l {OUT}/*.jsonl